# PCA and eigenfaces

We work with a face image library stored in `allFaces.mat`. Each grayscale face image is reshaped into a column vector, and we use **principal component analysis (PCA)** through the **singular value decomposition (SVD)** to build a low-dimensional coordinate system of **eigenfaces**.

The main goals are:

1. build the mean face and the mean-subtracted data matrix,
2. compute the eigenfaces from the SVD,
3. inspect the singular values and cumulative explained variance,
4. reconstruct a test face using different numbers of principal components,
5. visualize how images of different people separate in PCA coordinates,
6. perform a very simple recognition experiment in eigenface space.


## Learning goals

By the end of this notebook, you should be able to:

- load and organize a face image dataset for PCA,
- compute the mean face and mean-subtracted training matrix,
- compute eigenfaces from the economy SVD,
- interpret singular values in terms of variance and dimensionality reduction,
- reconstruct a face using a truncated PCA basis,
- project face images into a low-dimensional PCA space,
- understand why PCA can help with simple face recognition.


## 1. Load the face dataset and visualize a few images

We use the file `allFaces.mat`, which is placed in the local `data/` folder.

The dataset contains:

- `faces`: a matrix whose columns are vectorized face images,
- `n` and `m`: the image height and width,
- `nfaces`: the number of images available for each person.

Each column of `faces` is one aligned grayscale image.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

def load_face_data():
    mat = loadmat("data/allFaces.mat")
    faces = np.asarray(mat["faces"], dtype=float)
    n = int(np.array(mat["n"]).squeeze())
    m = int(np.array(mat["m"]).squeeze())
    nfaces = np.asarray(mat["nfaces"]).flatten().astype(int)

    return faces, n, m, nfaces

def face_image(vec, n, m):
    return vec.reshape((m, n)).T

def show_face(vec, n, m, ax=None, title=None, cmap="gray"):
    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(face_image(vec, n, m), cmap=cmap)
    ax.axis("off")
    if title is not None:
        ax.set_title(title)

def person_ranges(nfaces):
    starts = np.cumsum(np.concatenate(([0], nfaces[:-1])))
    stops = np.cumsum(nfaces)
    return list(zip(starts, stops))


In [ ]:
faces, n, m, nfaces = load_face_data()

print("faces shape:", faces.shape)
print("image size:", n, "x", m)
print("number of people:", len(nfaces))
print("first 10 values of nfaces:", nfaces[:10])


In [ ]:
# TODO: show one image for each of the first 9 people
# Hint:
# ranges = person_ranges(nfaces)
# fig, axes = plt.subplots(3, 3, figsize=(9, 9))


## 2. Build the training set and compute the mean face

We use the **first 36 people** as training data and keep the remaining two people as a small test set.

We then compute the average face of the training set and subtract it from every training image.


In [ ]:
# TODO: build the training matrix from the first 36 people
# TODO: compute the mean face and the mean-subtracted data matrix

# n_train = ...
# training_faces = ...
# avg_face = ...
# X = ...


In [ ]:
# TODO: plot the average face


## 3. Compute PCA from the SVD

We compute the economy SVD of the mean-subtracted training matrix:
$$
X = U\Sigma V^T.
$$

In this setting:

- the columns of $U$ are the **eigenfaces**,
- the singular values indicate the importance of each component,
- the columns of $V$ contain the PCA coordinates of the training images, up to scaling by $\Sigma$.


In [ ]:
# TODO: compute the economy SVD of X

# U, s, VT = ...


## 4. Visualize the first eigenfaces

The first few eigenfaces describe dominant patterns of variation in the training images.

They are not actual faces, but basis images that span the dominant directions of variation in the dataset.


In [ ]:
# TODO: display the first 8 eigenfaces
# Hint: use U[:, j] and reshape it into an image.


## 5. Singular values and cumulative explained variance

We now inspect how quickly the singular values decay and how much variance is captured by the first principal components.

This helps us understand why a relatively small number of components can already provide a useful approximation of a face.


In [ ]:
# TODO: compute and plot:
# - the singular values
# - the cumulative explained variance based on s**2


## 6. Reconstruct a test face with different numbers of components

We now choose a test image from a person **outside the training set** and reconstruct it using truncated PCA bases of different sizes.

If $U_r$ contains the first $r$ eigenfaces, then the rank-$r$ reconstruction is
$$
x_{\mathrm{recon}} = \bar{x} + U_r U_r^T (x_{\mathrm{test}} - \bar{x}).
$$


In [ ]:
# TODO: choose one test image from person 37 or 38
# TODO: subtract the mean face and reconstruct it for several values of r

# test_idx = ...
# test_face = ...
# test_face_ms = ...
# r_values = [25, 50, 100, 200, 400]


In [ ]:
# TODO: plot the original test face and its PCA reconstructions


## 7. Project two people onto a two-dimensional PCA space

Images of different people may separate in a low-dimensional PCA projection.

We reproduce this idea by taking two specific people and projecting all of their images onto the **5th and 6th principal components**.


In [ ]:
# TODO: choose two people, for example person 2 and person 7
# TODO: project all of their images onto PCA modes 5 and 6


In [ ]:
# TODO: make a 2D scatter plot of the PCA coordinates
# Use different markers/colors for the two people.


## 8. A simple recognition experiment in eigenface space

We perform a very simple recognition experiment.

We use the **first 6 people** only. For each person:
- all but the last image are used for training,
- the last image is used as a test image.

We compute a PCA basis from all training images, project everything into the first $r$ principal components, and classify each test image by the nearest class centroid in PCA space.


In [ ]:
# TODO: build a small training/test split using the first 6 people
# TODO: compute PCA coordinates
# TODO: classify each test image using the nearest centroid in PCA space
# A reasonable choice is r = 25.


## 9. Approximate a dog image using the eigenface basis

An eigenface basis, although trained on faces, can also be used to approximate a non-face image.

We reproduce that idea here using the dog image `dog.jpg`, stored in the `data/` folder.

The image is **not** part of the face training set. We convert it to grayscale, resize it to the same dimensions as the face images, subtract the average face, and then project it onto the eigenface basis.


In [ ]:
# Load the dog image, convert it to grayscale, and resize it
img = Image.open("data/dog.jpg").convert("L")
img = img.resize((m, n))

# Turn the image into a vector with the same format as the face data
dog = np.asarray(img, dtype=float).T.reshape(-1)
dog_ms = dog - avg_face

r_values_dog = [25, 100, 400, 800, 1600]


Reconstruct the dog image with several truncation ranks.

A good choice is
$$
r \in \{25, 100, 400, 800, 1600\}.
$$


In [ ]:
# TODO: plot the original dog image and its reconstructions

# Hint:
# recon = avg_face + U[:, :r] @ (U[:, :r].T @ dog_ms)


## Final discussion

1. Why do we subtract the mean face before applying PCA?
2. What do the first eigenfaces represent?
3. Why can a face often be approximated well with far fewer components than the number of pixels?
4. Why do images of different people sometimes separate in PCA space?
5. Why is a PCA basis trained on faces useful for recognition but not ideal for representing arbitrary non-face images?
6. What are the limitations of eigenfaces as a face-recognition method?
